## Day 8 — Batch Inference Pipeline
Score all 5.7M users with the registered model and save predictions to Gold Delta table.

**Run Cell 1 first after any cluster reset.**

In [0]:
# Cell 1 — Imports and Config
# Run this first every session — restores all variables after cluster reset
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import (
    col, round, when, current_timestamp, lit, desc, avg, count
)
import pyspark.sql.functions as F
import mlflow.spark

CATALOG      = 'ecommerce'
# Full 3-level Unity Catalog path — required since Day 7 fix
MODEL_NAME   = 'ecommerce.default.ecom_purchase_model'
MODEL_PATH   = '/Volumes/ecommerce/sc_ecommerce/vol_ecommerce/models/'
# TMP_DIR required for all mlflow.spark operations on serverless clusters
TMP_DIR      = '/Volumes/ecommerce/sc_ecommerce/vol_ecommerce/mlflow_tmp'
OUTPUT_TABLE = f'{CATALOG}.gold.user_predictions'

FEATURE_COLS = [
    'total_events', 'total_sessions', 'total_views',
    'total_cart_adds', 'avg_price_viewed'
]

# Evaluator for quick AUC verification
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='will_purchase',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)

print('Config ready')
print(f'Model    : {MODEL_NAME}')
print(f'Output   : {OUTPUT_TABLE}')


### Step 1 — Load the Trained Model
Try loading from MLflow Model Registry first (the official version registered in Day 7). Falls back to Volume path if registry load fails.

In [0]:
# Cell 2 — Load the registered model
# Option A: from MLflow Model Registry (official — registered in Day 7)
# Option B: directly from Volume path (fallback — always works)
try:
    model = mlflow.spark.load_model(
        model_uri  = f'models:/{MODEL_NAME}/1',
        dfs_tmpdir = TMP_DIR      # required on serverless clusters
    )
    print('Loaded from Model Registry')
except Exception as e:
    print(f'Registry load failed: {e}')
    print('Loading from Volume path instead...')
    model = PipelineModel.load(f'{MODEL_PATH}rf_best_tuned')
    print('Loaded from Volume path')

print(f'\nPipeline stages:')
for i, s in enumerate(model.stages):
    print(f'  Stage {i}: {type(s).__name__}')
# Expected:
#   Stage 0: VectorAssembler
#   Stage 1: RandomForestClassificationModel


### Step 2 — Load ALL Users
In Days 6-7 we used only `df_test` for evaluation. Today we score ALL 5.7M users — both train and test combined.

In [0]:
# Cell 3 — Load all 5.7M users from Gold ml_dataset
# We use ml_dataset because it already has the 5 feature columns
# and the will_purchase label (useful for accuracy check after scoring)
df_all_users = spark.table(f'{CATALOG}.gold.ml_dataset')

total_users = df_all_users.count()
print(f'Total users to score: {total_users:,}')
print(f'Columns: {df_all_users.columns}')

# Null check — model will silently skip rows with nulls (handleInvalid='skip')
# Better to know upfront if any feature has nulls
print('\nNull check:')
for c in FEATURE_COLS:
    n = df_all_users.filter(col(c).isNull()).count()
    status = f'WARNING: {n} nulls' if n > 0 else 'OK'
    print(f'  {c:<25} {status}')


### Step 3 — Run Batch Inference (Task 1)
`model.transform()` applies the full pipeline to every user. Spark distributes this automatically — no extra setup needed.

In [0]:
# Cell 4 - Run batch inference on all 5.7M users
# .cache() is NOT supported on serverless — removed
# Serverless manages memory automatically, no manual caching needed
print("Running batch inference on all users...")

df_scored = model.transform(df_all_users)

scored_count = df_scored.count()
print(f"Scoring complete!")
print(f"Scored rows: {scored_count:,}")
print(f"
New columns added by model:")
print("  rawPrediction - raw model scores before threshold")
print("  probability   - [prob_not_buy, prob_buy] as a vector")
print("  prediction    - 0.0 or 1.0 based on 0.5 threshold")

display(df_scored.select(
    "will_purchase", "prediction", "probability",
    "total_events", "total_cart_adds"
).limit(5))


### Step 4 — Extract Purchase Probability
`probability` is a vector like `[0.15, 0.85]`. Index 0 = probability of NOT buying. Index 1 = probability of buying. We extract index 1 as a clean number.

In [0]:
# Cell 5 — Extract the buy probability from the vector column
# vector_to_array converts the vector to an array so we can index into it
# [1] = index 1 = probability of buying (index 0 = not buying)
df_with_prob = df_scored.withColumn(
    'purchase_probability',
    round(vector_to_array(col('probability'))[1], 4)
)

# Add a simple true/false flag
# predicted_buyer = True if model thinks probability > 50%
df_with_prob = df_with_prob.withColumn(
    'predicted_buyer',
    when(col('purchase_probability') > 0.5, True).otherwise(False)
)

print('Probability extracted. Sample:')
display(df_with_prob.select(
    'will_purchase', 'prediction',
    'purchase_probability', 'predicted_buyer'
).limit(10))


### Step 5 — Build the Final Predictions Table (Task 2)
Select only useful columns, add metadata: when the prediction was made and which model produced it.

In [0]:
# Cell 6 — Select final columns for the Gold predictions table
df_predictions = df_with_prob.select(

    # User behaviour features (context for analysts)
    col('total_events'),
    col('total_sessions'),
    col('total_views'),
    col('total_cart_adds'),
    col('avg_price_viewed'),

    # Actual outcome — did they buy? (used for accuracy monitoring)
    col('will_purchase').alias('actual_purchase'),

    # Model output
    col('prediction').cast('integer').alias('predicted_purchase'),
    col('purchase_probability'),
    col('predicted_buyer'),

    # Metadata — who predicted this and when
    current_timestamp().alias('scored_at'),
    lit('rf_best_tuned_v1').alias('model_version')
)

print('Final predictions table schema:')
df_predictions.printSchema()
print(f'Row count: {df_predictions.count():,}')


### Step 6 — Save to Gold Delta Table
Write to `ecommerce.gold.user_predictions`. This is the permanent production output of your entire AI pipeline.

In [0]:
# Cell 7 - Save predictions to Gold Delta table
# saveAsTable() triggers PERSIST TABLE which is blocked on serverless
# Fix: use writeTo().createOrReplace() - fully serverless compatible

(
    df_predictions
    .writeTo(OUTPUT_TABLE)
    .using("delta")
    .createOrReplace()
)

saved_count = spark.table(OUTPUT_TABLE).count()
print(f"Saved to : {OUTPUT_TABLE}")
print(f"Rows     : {saved_count:,}")

display(spark.table(OUTPUT_TABLE).limit(5))


### Step 7 — Find Top Predicted Buyers (Task 3)
Sort all users by `purchase_probability` descending. Highest probability = most likely buyers = who marketing should target first.

In [0]:
# Cell 8 - Identify top predicted buyers
df_preds = spark.table(OUTPUT_TABLE)

# Top 20 users ranked by purchase probability
# Use string column name directly — avoids conflict with pyspark col() function
print("=== TOP 20 PREDICTED BUYERS ===")
display(
    df_preds
    .filter("predicted_buyer = true")
    .orderBy(desc("purchase_probability"))
    .select(
        "purchase_probability", "predicted_purchase", "actual_purchase",
        "total_cart_adds", "total_events", "total_sessions"
    )
    .limit(20)
)

# Summary counts
total     = df_preds.count()
predicted = df_preds.filter("predicted_buyer = true").count()
pct       = round(predicted / total * 100, 2) if total > 0 else 0.0

print(f"
Total users scored   : {total:,}")
print(f"Predicted buyers     : {predicted:,}  ({pct}%)")
print(f"Predicted non-buyers : {total - predicted:,}")


### Step 8 — Probability Band Analysis
Break users into probability bands and check the actual purchase rate per band. The buy rate **must** rise from band 1 to band 5 — that proves the model is genuinely ranking buyers higher.

In [0]:
# Cell 9 — Probability band analysis + confusion matrix
df_preds = spark.table(OUTPUT_TABLE)

# ── Confusion Matrix ──────────────────────────────────────────────────
# Shows how many the model got right vs wrong across all 5.7M users
print('=== CONFUSION MATRIX (ALL USERS) ===')
print('actual=1, predicted=1 -> correctly caught buyer     (True Positive)')
print('actual=1, predicted=0 -> missed a buyer             (False Negative)')
print('actual=0, predicted=1 -> wrongly flagged as buyer   (False Positive)')
print('actual=0, predicted=0 -> correctly ignored non-buyer (True Negative)')
df_preds.groupBy('actual_purchase', 'predicted_purchase') \
    .count() \
    .orderBy('actual_purchase', 'predicted_purchase') \
    .show()

# ── Probability Bands ─────────────────────────────────────────────────
# actual_buy_rate_% should INCREASE from band 1 to band 5
# If it does: model is working correctly
# If it doesn't: model has a calibration problem
df_bands = df_preds.withColumn('prob_band',
    when(col('purchase_probability') < 0.2, '1 — 0-20%   very unlikely')
    .when(col('purchase_probability') < 0.4, '2 — 20-40%  unlikely')
    .when(col('purchase_probability') < 0.6, '3 — 40-60%  uncertain')
    .when(col('purchase_probability') < 0.8, '4 — 60-80%  likely')
    .otherwise(                              '5 — 80-100% very likely')
)

print('=== USERS BY PROBABILITY BAND ===')
display(
    df_bands.groupBy('prob_band')
    .agg(
        count('*').alias('user_count'),
        round(avg('actual_purchase') * 100, 1).alias('actual_buy_rate_%')
    )
    .orderBy('prob_band')
)

# ── Average probability by actual outcome ─────────────────────────────
print('=== AVG PROBABILITY BY ACTUAL OUTCOME ===')
print('Actual buyers should have a much higher avg probability than non-buyers')
df_preds.groupBy('actual_purchase') \
    .agg(round(avg('purchase_probability'), 4).alias('avg_probability')) \
    .orderBy('actual_purchase') \
    .show()


### Day 8 Complete!

**What you built:**
- Loaded the registered model from MLflow (with `dfs_tmpdir` fix for serverless)
- Scored all 5.7M users in one batch inference run
- Extracted purchase probability from the model's output vector
- Saved results to `ecommerce.gold.user_predictions` (Gold Delta table)
- Identified the top predicted buyers ranked by probability
- Validated model quality with probability band analysis

**Next:** Day 9 — End-to-End Pipeline (automate the full Bronze to Predictions flow)